In [1]:
pip install transformers torch torchvision torchaudio pandas scikit-learn openpyxl tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 84.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 64.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 48.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 29.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 69.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.3/564.3 kB 36.5 MB/s eta 0:00:00
  Attempting uninstall:

In [2]:
pip install -q seqeval==1.2.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
Note: you may need to restart the kernel to use updated packages.


In [3]:

import gc, torch
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

In [4]:
# train_deberta_from_instruction_spans_with_per_type_metrics_and_loss.py
import os, re, json, warnings, random
from typing import List, Dict
from inspect import signature

import numpy as np
import pandas as pd

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_fscore_support

import transformers
from transformers import (
    DebertaTokenizerFast,
    DebertaForTokenClassification,
    Trainer,
    TrainingArguments,
    TrainerCallback,
)

try:
    from transformers import DataCollatorForTokenClassification
except Exception:
    DataCollatorForTokenClassification = None

warnings.filterwarnings("ignore")
print("Transformers version:", transformers.__version__)

# ========= Config =========
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

MODEL_NAME = "microsoft/deberta-base"
EXCEL_PATH = "/kaggle/input/dataset-db/improved_annotated_ner_final_no_overlap.xlsx"  # <-- set your path

MAX_LEN = 256
BATCH_TRAIN = 8
BATCH_EVAL = 16
NUM_EPOCHS = 15            # <-- change to 3/5/etc as needed
LR = 5e-5
TEST_SIZE = 0.2
OUTPUT_DIR = "./deberta_Anikait_out"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ========= Columns =========
TEXT_COL = "Instruction"
TYPE_COLS = ["ACTION","QUANTITY","UNIT","INGREDIENT","TOOL","TEMPERATURE","STATE","GARNISH"]

# ========= Sentence -> BIO conversion =========
def word_tokenize(s: str):
    return re.findall(r"\w+(?:'\w+)?|[^\w\s]", s)

def split_list_cell(cell):
    if pd.isna(cell): return []
    s = str(cell).strip()
    parts = re.split(r"\s*\|\s*|\s*,\s*|\s*;\s*", s)
    return [p for p in (p.strip() for p in parts) if p]

def normalize_tokens(toks):
    return [t.lower() for t in toks if re.search(r"\w", t)]

def find_and_tag(base_tokens, labels, phrase_tokens, tag):
    if not phrase_tokens: return
    n, m = len(base_tokens), len(phrase_tokens)
    i = 0
    while i <= n - m:
        if base_tokens[i:i+m] == phrase_tokens:
            labels[i] = f"B-{tag}"
            for k in range(1, m):
                labels[i+k] = f"I-{tag}"
            i += m
        else:
            i += 1

def load_excel_to_tokens_labels(path: str) -> pd.DataFrame:
    df = pd.read_excel(path)
    missing = [c for c in [TEXT_COL, *TYPE_COLS] if c not in df.columns]
    if missing:
        raise ValueError(f"Missing expected columns: {missing}")
    rows = []
    for _, r in df.iterrows():
        text = str(r[TEXT_COL])
        sent_tokens_raw = word_tokenize(text)
        sent_tokens_norm = normalize_tokens(sent_tokens_raw)
        labels = ["O"] * len(sent_tokens_norm)
        for tcol in TYPE_COLS:
            for ph in split_list_cell(r.get(tcol, None)):
                ph_tok_norm = normalize_tokens(word_tokenize(ph))
                find_and_tag(sent_tokens_norm, labels, ph_tok_norm, tcol)
        rows.append({"tokens": sent_tokens_norm, "labels": labels})
    return pd.DataFrame(rows)

raw_df = load_excel_to_tokens_labels(EXCEL_PATH)
raw_df = raw_df[raw_df["tokens"].map(len) > 0].reset_index(drop=True)

# Labels
all_tags = set(t for seq in raw_df["labels"] for t in seq)
if "O" not in all_tags: all_tags.add("O")
label_list = sorted(all_tags, key=lambda x: (x=="O", x))
label2id: Dict[str,int] = {lab:i for i, lab in enumerate(label_list)}
id2label: Dict[int,str] = {i:lab for lab,i in label2id.items()}
entity_types = sorted({lab.split("-",1)[1] for lab in all_tags if lab!="O" and "-" in lab})

print(f"Loaded sentences: {len(raw_df)}")
print(f"Labels ({len(label_list)}): {label_list}")
print(f"Entity types: {entity_types}")

train_df, test_df = train_test_split(raw_df, test_size=TEST_SIZE, random_state=SEED, shuffle=True)

# ========= Dataset =========
class NERDataset(Dataset):
    def __init__(self, df, tokenizer, label2id, max_len):
        self.df = df.reset_index(drop=True)
        self.tok = tokenizer
        self.label2id = label2id
        self.max_len = max_len
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        words: List[str] = self.df.loc[idx, "tokens"]
        labs:  List[str] = self.df.loc[idx, "labels"]
        enc = self.tok(
            words,
            is_split_into_words=True,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_offsets_mapping=False
        )
        word_ids = enc.word_ids()
        labels = []
        prev_wid = None
        for wid in word_ids:
            if wid is None:
                labels.append(-100)
            elif wid != prev_wid:
                tag = labs[wid] if wid < len(labs) else "O"
                labels.append(self.label2id.get(tag, self.label2id["O"]))
                prev_wid = wid
            else:
                labels.append(-100)
        return {
            "input_ids": torch.tensor(enc["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(enc["attention_mask"], dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long)
        }

# ========= Tokenizer & Model =========
tokenizer = DebertaTokenizerFast.from_pretrained(MODEL_NAME, add_prefix_space=True)
model = DebertaForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

train_dataset = NERDataset(train_df, tokenizer, label2id, MAX_LEN)
eval_dataset  = NERDataset(test_df, tokenizer, label2id, MAX_LEN)
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer) if DataCollatorForTokenClassification else None

# ========= seqeval (optional) + fallback =========
try:
    from seqeval.metrics import (
        classification_report as seqeval_classification_report,
        f1_score as seqeval_f1,
        precision_score as seqeval_precision,
        recall_score as seqeval_recall,
    )
    _HAS_SEQEVAL = True
except Exception:
    _HAS_SEQEVAL = False

def _bio_to_spans(tags: List[str]):
    spans = []; start = None; et = None
    for i, t in enumerate(tags):
        if t == "O" or t is None:
            if et is not None: spans.append((start, i-1, et)); start=None; et=None
            continue
        if "-" in t: pref, typ = t.split("-", 1)
        else: pref, typ = "O", None
        if pref == "B":
            if et is not None: spans.append((start, i-1, et))
            start, et = i, typ
        elif pref == "I":
            if et is None or typ != et:
                if et is not None: spans.append((start, i-1, et))
                start, et = i, typ
        else:
            if et is not None: spans.append((start, i-1, et)); start=None; et=None
    if et is not None: spans.append((start, len(tags)-1, et))
    return spans

def _entity_scores(true_tag_seqs, pred_tag_seqs):
    TP=FP=FN=0
    for ttags, ptags in zip(true_tag_seqs, pred_tag_seqs):
        t_spans = {(s,e,typ) for (s,e,typ) in _bio_to_spans(ttags)}
        p_spans = {(s,e,typ) for (s,e,typ) in _bio_to_spans(ptags)}
        m = t_spans & p_spans
        TP += len(m); FP += len(p_spans - m); FN += len(t_spans - m)
    def prf(tp, fp, fn):
        p = tp/(tp+fp) if tp+fp>0 else 0.0
        r = tp/(tp+fn) if tp+fn>0 else 0.0
        f = 2*p*r/(p+r) if p+r>0 else 0.0
        return p,r,f
    p_micro,r_micro,f_micro = prf(TP,FP,FN)
    return {"precision": p_micro, "recall": r_micro, "f1": f_micro}

def compute_metrics_seqeval(eval_pred):
    logits, labels = eval_pred
    pred_ids = np.argmax(logits, axis=-1)
    true_tags_batch, pred_tags_batch = [], []
    for p_row, l_row in zip(pred_ids, labels):
        t_seq, p_seq = [], []
        for p, l in zip(p_row, l_row):
            if l == -100: continue
            t_seq.append(id2label[int(l)]); p_seq.append(id2label[int(p)])
        true_tags_batch.append(t_seq); pred_tags_batch.append(p_seq)
    if _HAS_SEQEVAL:
        f1 = seqeval_f1(true_tags_batch, pred_tags_batch)
        p  = seqeval_precision(true_tags_batch, pred_tags_batch)
        r  = seqeval_recall(true_tags_batch, pred_tags_batch)
        try:
            rep = seqeval_classification_report(true_tags_batch, pred_tags_batch)
            with open(os.path.join(OUTPUT_DIR,"seqeval_report.txt"),"w",encoding="utf-8") as f: f.write(rep)
        except Exception: pass
        return {"seqeval_precision": p, "seqeval_recall": r, "seqeval_f1": f1}
    else:
        s = _entity_scores(true_tags_batch, pred_tags_batch)
        return {"seqeval_precision": s["precision"], "seqeval_recall": s["recall"], "seqeval_f1": s["f1"]}

# ========= TrainingArguments (NO CHECKPOINTS) =========
def build_training_args():
    sig = signature(TrainingArguments.__init__).parameters
    def supports(name): return name in sig
    base = dict(
        output_dir=OUTPUT_DIR,
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=BATCH_TRAIN,
        per_device_eval_batch_size=BATCH_EVAL,
        learning_rate=LR,
        weight_decay=0.01,
        logging_steps=50,
        seed=SEED,
    )
    if supports("report_to"): base["report_to"] = "none"
    if supports("remove_unused_columns"): base["remove_unused_columns"] = False

    # --- changed: disable any checkpointing during training ---
    if supports("evaluation_strategy"): base["evaluation_strategy"] = "epoch"
    if supports("save_strategy"):       base["save_strategy"] = "no"          # <-- no checkpoints
    if supports("save_steps"):          base["save_steps"] = None             # ignored when save_strategy="no"
    if supports("save_total_limit"):    base["save_total_limit"] = None       # no rotating saves
    if supports("load_best_model_at_end"): base["load_best_model_at_end"] = False  # <-- don't load best (no checkpoints)
    if supports("metric_for_best_model"):  base["metric_for_best_model"] = None
    if supports("greater_is_better"):      base["greater_is_better"] = None
    if supports("overwrite_output_dir"):   base["overwrite_output_dir"] = True  # optional, avoid clashes with old runs
    # ----------------------------------------------------------

    filtered = {k:v for k,v in base.items() if supports(k)}
    return TrainingArguments(**filtered)

training_args = build_training_args()

# ========= Trainer =========
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics_seqeval,
)

# ========= Helpers for per-type metrics & losses =========
def collapse_to_type(tag: str) -> str:
    if tag == "O" or tag is None:
        return "O"
    return tag.split("-", 1)[1] if "-" in tag else "O"

@torch.no_grad()
def _predict_logits_and_labels(model, dataset, data_collator, batch_size=16):
    dl = DataLoader(dataset, batch_size=batch_size, shuffle=False, collate_fn=data_collator)
    device = next(model.parameters()).device
    model.eval()
    all_logits, all_labels = [], []
    for batch in dl:
        batch = {k: v.to(device) for k, v in batch.items()}
        out = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"])
        all_logits.append(out.logits.cpu())
        all_labels.append(batch["labels"].cpu())
    return torch.cat(all_logits, dim=0), torch.cat(all_labels, dim=0)

def _per_type_loss_precision_recall_f1(logits, labels, id2label, include_O=False):
    mask = labels != -100
    labels_valid = labels[mask]
    ce = F.cross_entropy(
        logits.view(-1, logits.size(-1)),
        torch.clamp(labels.view(-1), min=0),
        reduction="none",
    ).view(labels.size())
    ce_valid = ce[mask]

    gold_tags = [id2label[int(i)] for i in labels_valid]
    pred_ids = torch.argmax(logits, dim=-1)[mask]
    pred_tags = [id2label[int(i)] for i in pred_ids]

    gold_types = [collapse_to_type(t) for t in gold_tags]
    pred_types = [collapse_to_type(t) for t in pred_tags]

    sums = {}; counts = {}
    for loss_val, typ in zip(ce_valid.tolist(), gold_types):
        if typ == "O" and not include_O:
            continue
        sums[typ] = sums.get(typ, 0.0) + float(loss_val)
        counts[typ] = counts.get(typ, 0) + 1

    types = sorted(set([t for t in gold_types if t != "O"] + (["O"] if include_O else [])))
    out = {}
    for typ in types:
        y_true = [1 if t == typ else 0 for t in gold_types]
        y_pred = [1 if t == typ else 0 for t in pred_types]
        p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
        mean_loss = (sums.get(typ, 0.0) / max(counts.get(typ, 0), 1))
        out[typ] = {
            "precision": float(p),
            "recall": float(r),
            "f1": float(f1),
            "loss": float(mean_loss),
            "token_count": int(counts.get(typ, 0)),
            "positive_support": int(sum(y_true)),
        }
    return out

@torch.no_grad()
def compute_overall_eval_loss_manual(model, dataset, data_collator, batch_size=16):
    model.eval()
    dl = DataLoader(dataset, batch_size=batch_size, shuffle=False, collate_fn=data_collator)
    device = next(model.parameters()).device
    tot_loss = 0.0; tot_cnt = 0
    for batch in dl:
        batch = {k: v.to(device) for k, v in batch.items()}
        out = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"])
        logits = out.logits
        labels = batch["labels"]
        ce = F.cross_entropy(
            logits.view(-1, logits.size(-1)),
            torch.clamp(labels.view(-1), min=0),
            reduction="none",
        ).view(labels.size())
        mask = labels != -100
        ce = ce[mask]
        tot_loss += float(ce.sum().cpu().item())
        tot_cnt  += int(mask.sum().cpu().item())
    return (tot_loss / tot_cnt) if tot_cnt > 0 else float("nan")

# ========= Per-epoch per-TYPE (precision, recall, f1, loss) logger =========
class PerTypeMetricsAndLossLogger(TrainerCallback):
    def __init__(self, eval_dataset, id2label, data_collator, output_dir, batch_size=16, include_O=False):
        self.eval_dataset = eval_dataset
        self.id2label = id2label
        self.data_collator = data_collator
        self.output_dir = output_dir
        self.batch_size = batch_size
        self.include_O = include_O
        os.makedirs(self.output_dir, exist_ok=True)
        self.csv_path = os.path.join(self.output_dir, "per_type_metrics_by_epoch.csv")
        if not os.path.exists(self.csv_path):
            with open(self.csv_path, "w", encoding="utf-8") as f:
                f.write("epoch,type,precision,recall,f1,loss,token_count,positive_support\n")

    def on_epoch_end(self, args, state, control, **kwargs):
        model = kwargs["model"]
        epoch_num = int(round(state.epoch)) if state.epoch is not None else -1
        logits, labels = _predict_logits_and_labels(
            model=model,
            dataset=self.eval_dataset,
            data_collator=self.data_collator,
            batch_size=self.batch_size,
        )
        per_type = _per_type_loss_precision_recall_f1(
            logits=logits,
            labels=labels,
            id2label=self.id2label,
            include_O=self.include_O,
        )
        with open(self.csv_path, "a", encoding="utf-8") as f:
            for typ, d in sorted(per_type.items()):
                f.write(f"{epoch_num},{typ},{d['precision']:.6f},{d['recall']:.6f},{d['f1']:.6f},{d['loss']:.6f},{d['token_count']},{d['positive_support']}\n")
        print(f"[PerTypeMetricsAndLossLogger] Epoch {epoch_num} → logged to {self.csv_path}")

# Register callback BEFORE training
trainer.add_callback(
    PerTypeMetricsAndLossLogger(
        eval_dataset=eval_dataset,
        id2label=id2label,
        data_collator=data_collator,
        output_dir=OUTPUT_DIR,
        batch_size=BATCH_EVAL,
        include_O=False,  # set True if you also want 'O' rows
    )
)

# ========= Train (and save training_log.csv) =========
print(f"Train examples: {len(train_dataset)}, Eval examples: {len(eval_dataset)}")
train_output = trainer.train()

log_hist = pd.DataFrame(trainer.state.log_history)
log_csv = os.path.join(OUTPUT_DIR, "training_log.csv")
log_hist.to_csv(log_csv, index=False)
print(f"Saved training log to: {log_csv}")

# ========= Evaluate (entity-level, O ignored) =========
eval_metrics = trainer.evaluate()
print("\n=== Seqeval Overall (entity-level, O ignored) ===")
for k, v in eval_metrics.items():
    if k.startswith("eval_"):
        try: print(f"{k}: {float(v):.4f}")
        except Exception: print(f"{k}: {v}")
print(f"Final training loss (if available): {train_output.metrics.get('train_loss', 'N/A')}")
print(f"Eval loss: {eval_metrics.get('eval_loss', 'N/A')}")

# ========= Final per-type CSV (precision/recall/F1 + per-type loss) =========
# 1) Eval logits/labels
logits_t, labels_t = _predict_logits_and_labels(
    model=trainer.model,
    dataset=eval_dataset,
    data_collator=data_collator,
    batch_size=BATCH_EVAL,
)

# 2) Per-type metrics (incl. loss)
per_type_final = _per_type_loss_precision_recall_f1(
    logits=logits_t,
    labels=labels_t,
    id2label=id2label,
    include_O=False,
)

# 3) Rows for each type
rows = []
for col in entity_types:
    d = per_type_final.get(col, None)
    if d is None:
        rows.append({"column": col, "precision": 0.0, "recall": 0.0, "f1": 0.0, "loss": np.nan, "support_pos": 0})
    else:
        rows.append({
            "column": col,
            "precision": d["precision"],
            "recall":    d["recall"],
            "f1":        d["f1"],
            "loss":      d["loss"],
            "support_pos": d["positive_support"],
        })

# 4) Overall row (robust overall eval loss)
overall_loss = eval_metrics.get("eval_loss", np.nan)
if (overall_loss is None) or (isinstance(overall_loss, float) and np.isnan(overall_loss)):
    overall_loss = compute_overall_eval_loss_manual(
        model=trainer.model,
        dataset=eval_dataset,
        data_collator=data_collator,
        batch_size=BATCH_EVAL,
    )

rows.append({
    "column": "OVERALL_SEQEVAL",
    "precision": float(eval_metrics.get("eval_seqeval_precision", np.nan)),
    "recall":    float(eval_metrics.get("eval_seqeval_recall", np.nan)),
    "f1":        float(eval_metrics.get("eval_seqeval_f1", np.nan)),
    "loss":      float(overall_loss),
    "support_pos": np.nan,
})

per_col_df = pd.DataFrame(rows)
csv_path = os.path.join(OUTPUT_DIR, "per_type_metrics.csv")
per_col_df.to_csv(csv_path, index=False)
print(f"\nSaved final per-type metrics (with per-type loss) to: {csv_path}")
print(per_col_df)

# ========= Save model/tokenizer =========
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"\nModel and tokenizer saved to {OUTPUT_DIR}")


2025-10-09 10:44:41.732040: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1760006681.894818      37 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1760006681.944497      37 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Transformers version: 4.53.3
Loaded sentences: 10000
Labels (15): ['B-ACTION', 'B-GARNISH', 'B-INGREDIENT', 'B-QUANTITY', 'B-STATE', 'B-TEMPERATURE', 'B-TOOL', 'B-UNIT', 'I-ACTION', 'I-GARNISH', 'I-INGREDIENT', 'I-QUANTITY', 'I-TEMPERATURE', 'I-TOOL', 'O']
Entity types: ['ACTION', 'GARNISH', 'INGREDIENT', 'QUANTITY', 'STATE', 'TEMPERATURE', 'TOOL', 'UNIT']


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/559M [00:00<?, ?B/s]

Train examples: 8000, Eval examples: 2000


Step,Training Loss
50,0.621300
100,0.252400
150,0.159100
200,0.136700
250,0.130900
300,0.101300
350,0.093000
400,0.096200
450,0.077200
500,0.065000


[PerTypeMetricsAndLossLogger] Epoch 1 → logged to ./deberta_Anikait_out/per_type_metrics_by_epoch.csv
[PerTypeMetricsAndLossLogger] Epoch 2 → logged to ./deberta_Anikait_out/per_type_metrics_by_epoch.csv
[PerTypeMetricsAndLossLogger] Epoch 3 → logged to ./deberta_Anikait_out/per_type_metrics_by_epoch.csv
[PerTypeMetricsAndLossLogger] Epoch 4 → logged to ./deberta_Anikait_out/per_type_metrics_by_epoch.csv
[PerTypeMetricsAndLossLogger] Epoch 5 → logged to ./deberta_Anikait_out/per_type_metrics_by_epoch.csv
[PerTypeMetricsAndLossLogger] Epoch 6 → logged to ./deberta_Anikait_out/per_type_metrics_by_epoch.csv
[PerTypeMetricsAndLossLogger] Epoch 7 → logged to ./deberta_Anikait_out/per_type_metrics_by_epoch.csv
[PerTypeMetricsAndLossLogger] Epoch 8 → logged to ./deberta_Anikait_out/per_type_metrics_by_epoch.csv
[PerTypeMetricsAndLossLogger] Epoch 9 → logged to ./deberta_Anikait_out/per_type_metrics_by_epoch.csv
[PerTypeMetricsAndLossLogger] Epoch 10 → logged to ./deberta_Anikait_out/per_type_


=== Seqeval Overall (entity-level, O ignored) ===
eval_loss: 0.0488
eval_seqeval_precision: 0.9877
eval_seqeval_recall: 0.9877
eval_seqeval_f1: 0.9877
eval_runtime: 24.2827
eval_samples_per_second: 82.3630
eval_steps_per_second: 2.5940
Final training loss (if available): 0.021453411536415418
Eval loss: 0.04883924126625061

Saved final per-type metrics (with per-type loss) to: ./deberta_Anikait_out/per_type_metrics.csv
            column  precision    recall        f1      loss  support_pos
0           ACTION   0.995418  0.996179  0.995798  0.048301       2617.0
1          GARNISH   0.973022  0.987226  0.980072  0.169824        548.0
2       INGREDIENT   0.990981  0.994453  0.992713  0.050743       3425.0
3         QUANTITY   0.999067  1.000000  0.999533  0.111137       1071.0
4            STATE   0.983193  0.991525  0.987342  0.114526        118.0
5      TEMPERATURE   0.994012  0.994012  0.994012  0.066301        167.0
6             TOOL   0.992143  0.990021  0.991081  0.133416       

In [5]:
from pathlib import Path
import shutil

OUT = Path("./deberta_ner_out")
ckpts = sorted(OUT.glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[-1]))
print("Found:", [p.name for p in ckpts])

# keep only the last 2; delete older ones
keep_n = 2
for p in ckpts[:-keep_n]:
    print("Deleting", p)
    shutil.rmtree(p)


Found: ['checkpoint-500', 'checkpoint-1000', 'checkpoint-1500', 'checkpoint-2000', 'checkpoint-2500', 'checkpoint-3000', 'checkpoint-3500', 'checkpoint-4000', 'checkpoint-4500', 'checkpoint-5000', 'checkpoint-5500', 'checkpoint-6000', 'checkpoint-6500']
Deleting deberta_ner_out/checkpoint-500
Deleting deberta_ner_out/checkpoint-1000
Deleting deberta_ner_out/checkpoint-1500
Deleting deberta_ner_out/checkpoint-2000
Deleting deberta_ner_out/checkpoint-2500
Deleting deberta_ner_out/checkpoint-3000
Deleting deberta_ner_out/checkpoint-3500
Deleting deberta_ner_out/checkpoint-4000
Deleting deberta_ner_out/checkpoint-4500
Deleting deberta_ner_out/checkpoint-5000
Deleting deberta_ner_out/checkpoint-5500


In [6]:
# make_metric_plots_by_epoch.py
# One chart per metric (F1, Precision, Recall, Loss). Lines = entity types.
# - No grids
# - No title
# - Font size 12 and bold everywhere
# - Save PNGs at 300 dpi

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ====== INPUTS ======
CSV_PATH = "/kaggle/working/deberta_Anikait_out/per_type_metrics_by_epoch.csv"   # change if needed
OUT_DIR  = Path("./epoch_plots")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ====== LOAD & CLEAN ======
df = pd.read_csv(CSV_PATH)
df.columns = [c.lower() for c in df.columns]
df = df[df["type"].str.upper() != "OVERALL_SEQEVAL"].copy()
df["epoch"] = pd.to_numeric(df["epoch"], errors="coerce")
df = df.sort_values(["epoch","type"])

metrics = [
    ("f1",        "F1 Score"),
    ("precision", "Precision"),
    ("recall",    "Recall"),
    ("loss",      "Loss"),
]

# ====== GLOBAL TEXT STYLE: size 12, bold ======
plt.rcParams.update({
    "font.size": 12,
    "font.weight": "bold",
    "axes.labelweight": "bold",
    "axes.titleweight": "bold",  # (no titles anyway, harmless)
})

def bold_ticklabels(ax):
    for lbl in ax.get_xticklabels() + ax.get_yticklabels():
        lbl.set_fontsize(12)
        lbl.set_fontweight("bold")

# ====== PLOT (one figure per metric) ======
for col, label in metrics:
    fig = plt.figure(figsize=(8, 5))
    ax = plt.gca()

    for typ, grp in df.groupby("type"):
        ax.plot(
            grp["epoch"], grp[col],
            marker="o", linewidth=2, markersize=5,  # no colors specified
            label=typ
        )

    # No title (as requested)
    ax.set_xlabel("Epoch", fontweight="bold", fontsize=12)
    ax.set_ylabel(label, fontweight="bold", fontsize=12)
    ax.grid(False)
    bold_ticklabels(ax)

    leg = ax.legend(loc="best", fontsize=12, ncols=2 if len(df["type"].unique()) > 6 else 1)
    for txt in leg.get_texts():
        txt.set_fontweight("bold")
        txt.set_fontsize(12)

    fig.tight_layout()
    out_path = OUT_DIR / f"{col}_by_epoch.png"
    fig.savefig(out_path, dpi=300, bbox_inches="tight")  # 300 dpi
    plt.close(fig)

print(f"Saved 300 dpi plots to: {OUT_DIR.resolve()}")


Saved 300 dpi plots to: /kaggle/working/epoch_plots


In [14]:
import os, re, shutil, glob

OUTPUT_DIR = "./deberta_ner_out"  # change if different

# find all checkpoint dirs and sort by step number
ckpts = sorted(
    glob.glob(os.path.join(OUTPUT_DIR, "checkpoint-*")),
    key=lambda p: int(re.search(r"checkpoint-(\d+)$", p).group(1))
)

print("Found checkpoints:", [os.path.basename(p) for p in ckpts])

# choose last 3 (newest by step)
to_delete = ckpts[-3:] if len(ckpts) >= 3 else ckpts
print("Deleting:", [os.path.basename(p) for p in to_delete])

for p in to_delete:
    shutil.rmtree(p)
print("Done.")


Found checkpoints: ['checkpoint-6000', 'checkpoint-6500', 'checkpoint-7000', 'checkpoint-7500']
Deleting: ['checkpoint-6500', 'checkpoint-7000', 'checkpoint-7500']
Done.
